# 07 — Gold: FactProductStock (Periodic Snapshot)

Grain: uma linha por `(ProductID, data_de_execução)`.

**Padrão Kimball — Periodic Snapshot:** cada execução adiciona um novo registro com o estado atual do estoque. Não há UPDATE — cada snapshot é imutável.

**Técnica DuckDB:** `INSERT INTO ... WHERE NOT EXISTS` — append-only, sem duplicar o snapshot do mesmo dia.

**Diferença dos outros fatos:**
- `FactSales`: imutável por design (transação passada)
- `FactOrderFulfillment`: linha é atualizada quando status muda (accumulating)
- `FactProductStock`: nova linha por execução, histórico preservado (periodic snapshot)

In [1]:
import sys, os
from datetime import date
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Periodic Snapshot — inserir estado atual do estoque
# Uma linha por produto por dia de execução
# ============================================================
today    = date.today()
snap_key = int(today.strftime("%Y%m%d"))  # ex: 20260308

print(f"SnapshotDateKey: {snap_key}")

# Verificar se snapshot de hoje já existe
existing = conn.execute(f"SELECT COUNT(*) FROM gold.FactProductStock WHERE SnapshotDateKey = {snap_key}").fetchone()[0]

if existing > 0:
    print(f"Snapshot de hoje ({snap_key}) já existe ({existing} linhas). Nenhuma ação.")
else:
    conn.execute(f"""
        INSERT INTO gold.FactProductStock
        SELECT
            CAST(hash(CAST({snap_key} AS VARCHAR) || ',' || CAST(dp.ProductSK AS VARCHAR)) % 2147483647 AS INTEGER) AS StockSK,
            {snap_key}                                        AS SnapshotDateKey,
            dp.ProductSK,
            dc.CategorySK,
            p.UnitsInStock,
            p.UnitsOnOrder,
            p.ReorderLevel,
            -- NeedsReorder: estoque atual <= nível de reposição
            p.UnitsInStock <= p.ReorderLevel                  AS NeedsReorder,
            current_timestamp                                  AS LoadTimestamp
        FROM bronze.products p
        JOIN gold.DimProduct  dp ON p.ProductID   = dp.ProductID  AND dp.IsCurrent = TRUE
        JOIN gold.DimCategory dc ON dp.CategoryName = dc.CategoryName
    """)
    n = conn.execute(f"SELECT COUNT(*) FROM gold.FactProductStock WHERE SnapshotDateKey = {snap_key}").fetchone()[0]
    print(f"Snapshot inserido: {snap_key} | {n} produtos")

SnapshotDateKey: 20260329
Snapshot inserido: 20260329 | 77 produtos


In [3]:
# ============================================================
# Validações
# ============================================================
print("1. Snapshots disponíveis:")
print(conn.execute("""
    SELECT SnapshotDateKey, COUNT(*) AS Produtos
    FROM gold.FactProductStock
    GROUP BY SnapshotDateKey ORDER BY SnapshotDateKey
""").fetchdf().to_string(index=False))

print("\n2. Produtos com NeedsReorder = True (último snapshot):")
max_snap = conn.execute("SELECT MAX(SnapshotDateKey) FROM gold.FactProductStock").fetchone()[0]
print(f"   Último snapshot: {max_snap}")
print(conn.execute(f"""
    SELECT
        p.ProductName,
        p.CategoryName,
        fs.UnitsInStock,
        fs.ReorderLevel,
        fs.UnitsOnOrder,
        fs.UnitsInStock - fs.ReorderLevel AS EstoqueAcimaReorder
    FROM gold.FactProductStock fs
    JOIN gold.DimProduct p ON fs.ProductSK = p.ProductSK
    WHERE fs.SnapshotDateKey = {max_snap}
      AND fs.NeedsReorder = TRUE
    ORDER BY EstoqueAcimaReorder
""").fetchdf().to_string(index=False))

print("\n3. NeedsReorder — resumo:")
print(conn.execute(f"""
    SELECT NeedsReorder, COUNT(*) AS Qtd
    FROM gold.FactProductStock
    WHERE SnapshotDateKey = {max_snap}
    GROUP BY NeedsReorder
""").fetchdf().to_string(index=False))

print("\n4. Grain único (SnapshotDateKey, ProductSK):")
dups = conn.execute("""
    SELECT COUNT(*) FROM (
        SELECT SnapshotDateKey, ProductSK
        FROM gold.FactProductStock
        GROUP BY SnapshotDateKey, ProductSK HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"   Duplicatas: {dups} (esperado: 0)")

1. Snapshots disponíveis:
 SnapshotDateKey  Produtos
        20260329        77

2. Produtos com NeedsReorder = True (último snapshot):
   Último snapshot: 20260329
              ProductName   CategoryName  UnitsInStock  ReorderLevel  UnitsOnOrder  EstoqueAcimaReorder
        Gorgonzola Telino Dairy Products             0            20            70                  -20
       Mascarpone Fabioli Dairy Products             9            25            40                  -16
Louisiana Hot Spiced Okra     Condiments             4            20           100                  -16
            Outback Lager      Beverages            15            30            10                  -15
               Gravad lax        Seafood            11            25            50                  -14
            Aniseed Syrup     Condiments            13            25            70                  -12
              Rogede sild        Seafood             5            15            70                  -10
   

In [4]:
# ============================================================
# LAB: Simular segundo snapshot (dia diferente)
# Demonstra o crescimento histórico da tabela
# ============================================================
fake_snap = 20260309  # Um dia depois

existing_fake = conn.execute(f"SELECT COUNT(*) FROM gold.FactProductStock WHERE SnapshotDateKey = {fake_snap}").fetchone()[0]
if existing_fake == 0:
    conn.execute(f"""
        INSERT INTO gold.FactProductStock
        SELECT
            CAST(hash(CAST({fake_snap} AS VARCHAR) || ',' || CAST(dp.ProductSK AS VARCHAR)) % 2147483647 AS INTEGER) AS StockSK,
            {fake_snap} AS SnapshotDateKey,
            dp.ProductSK, dc.CategorySK,
            p.UnitsInStock, p.UnitsOnOrder, p.ReorderLevel,
            p.UnitsInStock <= p.ReorderLevel AS NeedsReorder,
            current_timestamp AS LoadTimestamp
        FROM bronze.products p
        JOIN gold.DimProduct  dp ON p.ProductID    = dp.ProductID  AND dp.IsCurrent = TRUE
        JOIN gold.DimCategory dc ON dp.CategoryName = dc.CategoryName
    """)
    print(f"Snapshot simulado {fake_snap}: inserido")

print("\nSnapshots após segundo run:")
print(conn.execute("""
    SELECT SnapshotDateKey, COUNT(*) AS Produtos
    FROM gold.FactProductStock
    GROUP BY SnapshotDateKey ORDER BY SnapshotDateKey
""").fetchdf().to_string(index=False))

conn.close()

Snapshot simulado 20260309: inserido

Snapshots após segundo run:
 SnapshotDateKey  Produtos
        20260309        77
        20260329        77
